# Machine Learning–Driven Content Refresh Prioritization

**FlyRank ML Internship — Capstone | Lane 2: Refresh / Content Opportunity Scoring**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shoriful-mynul/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

### Executive summary

This capstone builds a leakage-aware machine-learning ranking system to prioritize mature content pages for review or refresh. The system uses **March 2026** search and engagement signals available at decision time and evaluates them against an **April 2026** future decline outcome.

The final deliverable is a **ranked action queue** containing a decline probability, transparent reason codes, and a recommended review action.

> **Important:** this is a decision-support ranking system, not a causal model and not a guarantee that a refresh will recover traffic.

In [ ]:
# Core setup and public-safe warehouse access

import os
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from huggingface_hub import get_token

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

hf_token = get_token()
if not hf_token:
    raise RuntimeError("A Hugging Face READ token is required to access the gated warehouse.")

con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MARCH_PATH = f"{FACT}/month=2026-03/data_0.parquet"
APRIL_PATH = f"{FACT}/month=2026-04/data_0.parquet"
CONTENT_PATH = f"{REL}/dim_content.parquet"
CLIENT_PATH = f"{REL}/dim_clients.parquet"

# Dimension tables are loaded explicitly so the notebook runs top-to-bottom from a fresh kernel.
content = con.read_parquet(CONTENT_PATH)
clients = con.read_parquet(CLIENT_PATH)

print("DuckDB:", duckdb.__version__)
print("Clients:", clients.shape)
print("Content:", content.shape)

## 1. Question & decision framing

**Research question:** Can historical search-performance and content signals help prioritize pages that are most worth reviewing or refreshing?

**Decision question:** Which existing content pages should a content team review first, using only information available before the review decision?

### Decision design

- **Decision window:** 1–31 March 2026
- **Future evaluation window:** 1–30 April 2026
- **Decision cutoff:** 31 March 2026
- **Maturity requirement:** content age ≥ 90 days
- **Demand requirement:** ≥ 500 March GSC impressions
- **Target:** April impressions ≤ 80% of March impressions, among pages with observable GSC data in both periods
- **Validation:** client-grouped holdout to avoid client leakage
- **Primary ranking metric:** Precision@50

In [ ]:
# Load the two monthly fact partitions and establish the analysis windows.

march = con.read_parquet(MARCH_PATH)
april = con.read_parquet(APRIL_PATH)

print("March:", march.shape, march["report_date"].min(), "to", march["report_date"].max())
print("April:", april.shape, april["report_date"].min(), "to", april["report_date"].max())

qa = pd.DataFrame({
    "check": [
        "March rows",
        "March unique dates",
        "March unique client-page pairs",
        "April rows",
        "April unique dates",
        "March pages with GSC impressions",
        "April rows with GSC availability",
        "March-to-content unmatched rows",
    ],
    "value": [
        len(march),
        march["report_date"].nunique(),
        march[["client_hash_id","content_hash_id"]].drop_duplicates().shape[0],
        len(april),
        april["report_date"].nunique(),
        march.loc[march["gsc_impressions"] > 0, "content_hash_id"].nunique(),
        int((april["gsc_data_available"] == True).sum()),
        int(len(march.merge(content[["client_hash_id","content_hash_id"]], on=["client_hash_id","content_hash_id"], how="left", indicator=True).query("_merge == 'left_only'"))),
    ],
})

display(qa)
print("March grain duplicate check:", march.duplicated(["client_hash_id","content_hash_id","report_date"]).sum())

## 2. Data & decision-time feature construction

The analysis uses the pseudonymized FlyRank internship warehouse:

- `dim_content` — content-page metadata
- `dim_clients` — client-level metadata
- `fact_content_daily_performance` — daily search and traffic performance

Only information available at the **31 March 2026** cutoff is used as model input. April is reserved for target construction and evaluation.

Client and content identifiers remain pseudonymized; no names, domains, private queries, credentials, or raw warehouse exports are included.

In [ ]:
# Build the March decision-time feature frame.

march_features = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_30d,
        SUM(gsc_clicks) AS clicks_30d,
        AVG(gsc_avg_position) AS avg_position_30d,
        SUM(ga4_pageviews) AS pageviews_30d,
        SUM(ga4_sessions) AS sessions_30d,
        SUM(ga4_users) AS users_30d,
        SUM(ga4_engaged_sessions) AS engaged_sessions_30d,
        SUM(sessions_ai) AS ai_sessions_30d,
        SUM(scroll_events) AS scroll_events_30d,
        COUNT(DISTINCT report_date) AS days_observed,
        MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS has_gsc_data,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data
    FROM march
    GROUP BY client_hash_id, content_hash_id
""").df()

feature_frame = con.sql("""
    SELECT
        m.*,
        c.content_created_date,
        c.content_updated_date,
        c.content_type,
        c.search_volume,
        c.competition,
        c.competition_level,
        c.cpc,
        c.main_intent,
        c.backlinks,
        c.category_count,
        c.char_count,
        c.word_count,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.is_published,
        c.is_deleted
    FROM march_features m
    INNER JOIN content c
        ON m.client_hash_id = c.client_hash_id
       AND m.content_hash_id = c.content_hash_id
""").df()

decision_date = pd.Timestamp("2026-03-31")
for col in ["content_created_date", "content_updated_date", "last_optimized_date", "optimization_eligible_date"]:
    feature_frame[col] = pd.to_datetime(feature_frame[col], errors="coerce")

feature_frame["content_age_days"] = (decision_date - feature_frame["content_created_date"]).dt.days
feature_frame["days_since_last_optimized"] = (decision_date - feature_frame["last_optimized_date"]).dt.days

eligible_features = feature_frame[feature_frame["content_created_date"] <= decision_date].copy()
model_population = eligible_features[eligible_features["content_age_days"] >= 90].copy()

model_population["ctr_30d"] = model_population["clicks_30d"] / model_population["impressions_30d"].replace(0, np.nan)
model_population["engagement_rate_30d"] = model_population["engaged_sessions_30d"] / model_population["sessions_30d"].replace(0, np.nan)

march_position = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_sum_position) AS sum_position_30d,
           SUM(gsc_impressions) AS impressions_for_position_30d
    FROM march
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
""").df()

march_position["weighted_avg_position_30d"] = march_position["sum_position_30d"] / march_position["impressions_for_position_30d"]

model_features = model_population.merge(
    march_position[["client_hash_id", "content_hash_id", "weighted_avg_position_30d"]],
    on=["client_hash_id", "content_hash_id"],
    how="left",
)
model_features.loc[model_features["weighted_avg_position_30d"] <= 0, "weighted_avg_position_30d"] = np.nan
model_features = model_features.drop(columns=["avg_position_30d"], errors="ignore")

print("Initial March feature frame:", march_features.shape)
print("Joined feature frame:", feature_frame.shape)
print("Decision-time eligible pages:", len(model_features))
print("Weighted-position missing:", model_features["weighted_avg_position_30d"].isna().sum())
print("CTR > 100%:", int((model_features["ctr_30d"] > 1).sum()))
print("Engagement rate > 100%:", int((model_features["engagement_rate_30d"] > 1).sum()))

## 3. Future-window target definition

A page is labeled **declining** when its April GSC impressions are at least **20% lower** than its March impressions.

The target is intentionally built from a future window. Pages without observable GSC data in both periods are excluded from supervised training because their outcome cannot be measured reliably.

This separates **decision-time features** from **future outcome information** and prevents the April target from leaking into the model inputs.

In [ ]:
# Construct and validate the future decline target.

april_target = con.sql("""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_april_30d,
           MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS has_april_gsc
    FROM april
    GROUP BY client_hash_id, content_hash_id
""").df()

target_frame = model_features.merge(april_target, on=["client_hash_id", "content_hash_id"], how="left")
target_frame["target_observable"] = (
    (target_frame["has_gsc_data"] == 1)
    & (target_frame["has_april_gsc"] == 1)
    & (target_frame["impressions_30d"] >= 500)
)
target_frame["is_declining"] = target_frame["target_observable"] & (target_frame["impressions_april_30d"] <= 0.80 * target_frame["impressions_30d"])

observable = target_frame[target_frame["target_observable"]].copy()
observable["impression_change_pct"] = ((observable["impressions_april_30d"] - observable["impressions_30d"]) / observable["impressions_30d"]) * 100
model_data = observable.copy()
model_data["target"] = model_data["is_declining"].astype(int)

target_summary = pd.DataFrame({
    "metric": ["Decision-time pages", "Observable future outcomes", "Declining pages", "Not declining pages", "Decline rate", "March median impressions", "April median impressions", "Median impression change"],
    "value": [len(target_frame), len(observable), int(model_data["target"].sum()), int((model_data["target"] == 0).sum()), model_data["target"].mean(), observable["impressions_30d"].median(), observable["impressions_april_30d"].median(), observable["impression_change_pct"].median()],
})
display(target_summary.round(4))

positive_check = (observable.loc[observable["target"] == 1, "impressions_april_30d"] <= 0.80 * observable.loc[observable["target"] == 1, "impressions_30d"]).all()
negative_check = (observable.loc[observable["target"] == 0, "impressions_april_30d"] <= 0.80 * observable.loc[observable["target"] == 0, "impressions_30d"]).sum()

print("Positive labels satisfy the decline rule:", positive_check)
print("Negative labels incorrectly satisfy the rule:", int(negative_check))

## 4. Transparent baseline

Before fitting ML models, a simple heuristic establishes a business-readable benchmark.

The baseline combines:

1. search visibility,
2. content age,
3. ranking-position opportunity, and
4. CTR weakness.

The baseline is evaluated on the same held-out client groups as the ML models.

In [ ]:
# Transparent rule-based baseline.

baseline = model_data.copy()
baseline["visibility_score"] = baseline["impressions_30d"].rank(pct=True)
baseline["freshness_risk_score"] = baseline["content_age_days"].rank(pct=True)
baseline["position_opportunity_score"] = baseline["weighted_avg_position_30d"].rank(pct=True, na_option="keep").fillna(0)
baseline["ctr_weakness_score"] = baseline["ctr_30d"].rank(pct=True, ascending=True)
baseline["baseline_score"] = 100 * (0.35 * baseline["visibility_score"] + 0.25 * baseline["freshness_risk_score"] + 0.25 * baseline["position_opportunity_score"] + 0.15 * baseline["ctr_weakness_score"])

baseline_ranked = baseline.sort_values("baseline_score", ascending=False).reset_index(drop=True)
baseline_precision_at_50 = baseline_ranked.head(50)["target"].mean()

print("Baseline Precision@50:", round(baseline_precision_at_50, 4))
print("Declining pages in baseline top 50:", int(baseline_ranked.head(50)["target"].sum()))
print("Overall positive rate:", round(baseline["target"].mean(), 4))

## 5. Leakage audit & ML feature contract

The model is restricted to signals that exist by the March decision cutoff.

**Excluded from model features:**

- April impressions
- the future decline label
- future-derived outcome fields
- any post-cutoff performance signal

The final feature set contains search performance, analytics performance, data-availability indicators, content age, and content metadata.

In [ ]:
# Explicit leakage audit.

feature_columns = [
    "impressions_30d", "clicks_30d", "weighted_avg_position_30d", "ctr_30d",
    "pageviews_30d", "sessions_30d", "users_30d", "engaged_sessions_30d",
    "engagement_rate_30d", "scroll_events_30d", "ai_sessions_30d",
    "days_observed", "has_gsc_data", "has_ga4_data", "content_age_days",
    "search_volume", "competition", "cpc", "backlinks", "category_count",
    "char_count", "word_count",
]

X = model_data[feature_columns].copy()
y = model_data["target"].astype(int)
groups = model_data["client_hash_id"]

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

missing_summary = X.isna().sum().to_frame("missing_count").assign(missing_pct=lambda d: d["missing_count"] / len(X) * 100).sort_values("missing_count", ascending=False)

leakage_audit = pd.DataFrame({
    "field": ["March performance features", "Content metadata available by cutoff", "April impressions", "Future decline target"],
    "allowed_as_feature": [True, True, False, False],
    "reason": ["Observed during the decision window", "Available at decision time", "Used only to construct the future outcome", "Outcome definition, not an input"],
})

display(leakage_audit)
display(missing_summary.head(10).round(2))
print("ML rows:", X.shape[0])
print("ML features:", X.shape[1])
print("Features with missing values:", int((missing_summary["missing_count"] > 0).sum()))

## 6. Client-aware modeling & validation

A random row split could place pages from the same client in both training and testing data. That would make performance look stronger than it would be on unseen clients.

Therefore, the model selection process uses **StratifiedGroupKFold** by `client_hash_id`, and the final reported test set is a complete client holdout.

Median imputation and scaling are fitted on training data only.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(sgkf.split(X, y, groups=groups))

fold_summary = []
for i, (train_idx, test_idx) in enumerate(splits, start=1):
    train_groups = groups.iloc[train_idx]
    test_groups = groups.iloc[test_idx]
    fold_summary.append({
        "fold": i, "train_rows": len(train_idx), "test_rows": len(test_idx),
        "train_clients": train_groups.nunique(), "test_clients": test_groups.nunique(),
        "client_overlap": len(set(train_groups) & set(test_groups)),
        "train_target_rate": y.iloc[train_idx].mean(), "test_target_rate": y.iloc[test_idx].mean(),
    })

fold_df = pd.DataFrame(fold_summary)
display(fold_df.round(4))

# Preserve the selected Fold 2 split used in the completed analysis.
train_idx, test_idx = splits[1]
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Final train rows:", len(X_train))
print("Final test rows:", len(X_test))
print("Final train clients:", groups_train.nunique())
print("Final test clients:", groups_test.nunique())
print("Client overlap:", len(set(groups_train) & set(groups_test)))

### Model comparison

Three standard classifiers are evaluated:

- Logistic Regression
- Decision Tree
- Random Forest

The selection criterion is the **ranking objective** (Precision@50), because the intended use is a limited review queue rather than a binary automated decision.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25, class_weight="balanced_subsample", random_state=42, n_jobs=-1),
}

evaluation_results = []
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    top_50_idx = np.argsort(y_prob)[::-1][:50]
    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
        "Average_Precision": average_precision_score(y_test, y_prob),
        "Precision@50": y_test.iloc[top_50_idx].mean(),
    })

evaluation_df = pd.DataFrame(evaluation_results).sort_values(["Precision@50", "Average_Precision", "ROC_AUC"], ascending=False).reset_index(drop=True)
display(evaluation_df.round(4))

rf_model = models["Random Forest"]
rf_prob = rf_model.predict_proba(X_test_processed)[:, 1]

In [ ]:
# Compare the selected model against the transparent baseline on the same holdout.

test_baseline = model_data.iloc[test_idx].copy()
test_baseline["visibility_score"] = test_baseline["impressions_30d"].rank(pct=True)
test_baseline["freshness_risk_score"] = test_baseline["content_age_days"].rank(pct=True)
test_baseline["position_opportunity_score"] = test_baseline["weighted_avg_position_30d"].rank(pct=True, na_option="keep").fillna(0)
test_baseline["ctr_weakness_score"] = test_baseline["ctr_30d"].rank(pct=True, ascending=True)
test_baseline["baseline_score"] = 100 * (0.35 * test_baseline["visibility_score"] + 0.25 * test_baseline["freshness_risk_score"] + 0.25 * test_baseline["position_opportunity_score"] + 0.15 * test_baseline["ctr_weakness_score"])

baseline_top_50 = test_baseline.sort_values("baseline_score", ascending=False).head(50)
comparison_df = pd.DataFrame({
    "Approach": ["Heuristic Baseline", "Random Forest"],
    "Precision@50": [baseline_top_50["target"].mean(), y_test.iloc[np.argsort(rf_prob)[::-1][:50]].mean()],
})
comparison_df["Lift_vs_overall_rate"] = comparison_df["Precision@50"] / y_test.mean()
display(comparison_df.round(4))
print("Overall test positive rate:", round(y_test.mean(), 4))

## 7. Final ranking, recommendations & artifacts

The selected Random Forest is used as a **ranking model** for the deployment-style scoring population.

The final queue uses only March decision-time information. April is not used for scoring.

Each ranked page receives:

- priority rank,
- decline probability,
- key observed signals,
- reason codes,
- a recommended review action.

The probability should be interpreted as a **model score for prioritization**, not as a guarantee of future decline or recovery.

In [ ]:
# Score the final March decision-time population.

scoring_population = model_features[(model_features["has_gsc_data"] == 1) & (model_features["impressions_30d"] >= 500)].copy()
X_score = scoring_population[feature_columns].copy()
for col in X_score.columns:
    X_score[col] = pd.to_numeric(X_score[col], errors="coerce")
X_score_processed = preprocessor.transform(X_score)
scoring_population["decline_probability"] = rf_model.predict_proba(X_score_processed)[:, 1]
scoring_population = scoring_population.sort_values("decline_probability", ascending=False).reset_index(drop=True)
scoring_population["priority_rank"] = scoring_population.index + 1

def generate_reason_codes(row):
    reasons = []
    if pd.notna(row["ctr_30d"]) and row["ctr_30d"] <= 0.01:
        reasons.append("Very low CTR")
    if row["impressions_30d"] >= 1000:
        reasons.append("High search visibility")
    if pd.notna(row["weighted_avg_position_30d"]) and row["weighted_avg_position_30d"] >= 10:
        reasons.append("Weak average search position")
    if row["content_age_days"] >= 180:
        reasons.append("Mature content")
    if pd.notna(row["engagement_rate_30d"]) and row["engagement_rate_30d"] < 0.5:
        reasons.append("Low engagement rate")
    if pd.notna(row["word_count"]) and row["word_count"] >= 2000:
        reasons.append("Long-form content")
    if not reasons:
        reasons.append("Multiple historical performance signals")
    return "; ".join(reasons[:4])

def recommend_action(row):
    reasons = row["reason_codes"]
    if "Very low CTR" in reasons:
        return "Review title/meta and search-intent alignment"
    if "Weak average search position" in reasons:
        return "Review topical relevance and on-page optimization"
    if "Low engagement rate" in reasons:
        return "Review content relevance and user experience"
    return "Prioritize for content refresh review"

scoring_population["reason_codes"] = scoring_population.apply(generate_reason_codes, axis=1)
scoring_population["recommended_action"] = scoring_population.apply(recommend_action, axis=1)

final_ranking = scoring_population[["priority_rank", "client_hash_id", "content_hash_id", "decline_probability", "impressions_30d", "clicks_30d", "ctr_30d", "weighted_avg_position_30d", "content_age_days", "reason_codes", "recommended_action"]].copy()

output_path = "final_content_refresh_ranking.csv"
final_ranking.to_csv(output_path, index=False)

display(final_ranking.head(10).round(4))
print("Final ranking rows:", len(final_ranking))
print("Final ranking columns:", len(final_ranking.columns))

In [ ]:
# Final artifact quality checks.

checks = {
    "Rank sequence valid": final_ranking["priority_rank"].equals(pd.Series(range(1, len(final_ranking) + 1))),
    "Duplicate client-page pairs": final_ranking.duplicated(["client_hash_id", "content_hash_id"]).sum(),
    "Missing decline probabilities": final_ranking["decline_probability"].isna().sum(),
    "Invalid probabilities": ((final_ranking["decline_probability"] < 0) | (final_ranking["decline_probability"] > 1)).sum(),
    "Missing reason codes": final_ranking["reason_codes"].isna().sum(),
    "Missing recommended actions": final_ranking["recommended_action"].isna().sum(),
    "Probability sorted descending": final_ranking["decline_probability"].is_monotonic_decreasing,
}

for k, v in checks.items():
    print(f"{k}: {v}")

print("Top probability:", round(final_ranking["decline_probability"].iloc[0], 4))
print("Bottom probability:", round(final_ranking["decline_probability"].iloc[-1], 4))
print("Exported:", output_path)

### Key visual evidence

The following charts summarize the most important evidence without turning the notebook into a dashboard.

1. target distribution,
2. baseline vs. Random Forest Precision@50,
3. model ranking comparison,
4. final decline-score distribution.

In [ ]:
# Portfolio-ready charts. All chart data are derived from objects created above.

fig, ax = plt.subplots(figsize=(6, 4))
target_counts = model_data["target"].value_counts().sort_index()
ax.bar(["Not declining", "Declining"], target_counts.values)
ax.set_title("Future Decline Target Distribution")
ax.set_ylabel("Pages")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
plot_df = comparison_df.copy()
ax.bar(plot_df["Approach"], plot_df["Precision@50"])
ax.set_title("Precision@50: Baseline vs Random Forest")
ax.set_ylabel("Precision@50")
ax.set_ylim(0, max(0.7, plot_df["Precision@50"].max() * 1.2))
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(evaluation_df["Model"], evaluation_df["Precision@50"])
ax.set_title("Model Ranking Performance")
ax.set_ylabel("Precision@50")
ax.set_ylim(0, max(0.7, evaluation_df["Precision@50"].max() * 1.2))
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(final_ranking["decline_probability"], bins=20)
ax.set_title("Final Decline-Probability Distribution")
ax.set_xlabel("Predicted decline probability")
ax.set_ylabel("Pages")
plt.tight_layout()
plt.show()

## Interpretation & limitations

### What the analysis supports

- Historical March signals can be used to create a ranked review queue for pages that subsequently met the defined April decline rule.
- The Random Forest achieved **Precision@50 = 0.54** on the selected client-held-out test set, compared with **0.40** for the transparent heuristic baseline.
- The final queue is therefore useful as a **prioritization mechanism**, especially when review capacity is limited.

### What it does not support

- It does **not** prove that refreshing a page will recover traffic.
- It does **not** predict Google algorithm changes.
- The April decline label is an observed outcome under a chosen threshold, not a causal intervention outcome.
- Conventional classification metrics are weak on the selected holdout, so the model should be treated as directional ranking support rather than a high-confidence automated classifier.
- Reason codes are rule-based explanations of observable signals; they are not causal explanations of model predictions.

### Reproducibility note

The notebook requires approved access to the gated FlyRank internship warehouse. The public repository contains the analysis code and pseudonymized outputs, not raw warehouse exports.

## Submission self-check

- [x] Research question and decision framing are explicit.
- [x] March decision window and April future window are explicit.
- [x] Future-window target is separated from decision-time features.
- [x] Client-aware validation is used.
- [x] Transparent baseline is reported.
- [x] Model comparison is reported on the same holdout.
- [x] Final ranking includes reason codes and recommended actions.
- [x] Final artifact quality checks are included.
- [x] Public-safe pseudonymized identifiers are used.
- [x] Claims use careful decision-support language.

**Final output:** `final_content_refresh_ranking.csv`